In [49]:
# imports and settings

import os
import gpxpy
import time
import torch
import librosa
import pickle
import warnings
import pandas as pd
import networkx as nx

import numpy as np
from numpy import linalg as LA
from numpy import histogram2d

from scipy import signal
from scipy.fft import fft, fftfreq, fftshift
from scipy.signal import find_peaks, butter, filtfilt, welch, get_window
from scipy.ndimage import gaussian_filter
from scipy.io import wavfile
from scipy.stats import wasserstein_distance_nd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.io import write_image

import utils as ut

# do not show warnings
warnings.filterwarnings("ignore")

# autoreload for development
%load_ext autoreload
%autoreload 2

print("Imports complete.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Imports complete.


---
## Vanilla

In [50]:
# WAV
folder_path = '../data/croatia/2407_1_600m'
file_list = os.listdir(folder_path)
file_list = [f for f in file_list if f.endswith('.wav')]
file_list.sort()

data = []
for file in file_list[6:-15]:
    # print(f"Loading file: {file}")
    _data, fs = librosa.load(os.path.join(folder_path, file))
    data.append(_data)
data = np.concatenate(data)
# data = data[80*fs:]
print(f"data final shape: {data.shape} of duration: {len(data)/fs} seconds")


# GPX
gpx_file = os.path.join(folder_path, '600m_route.gpx')
gpx_data = gpxpy.parse(open(gpx_file))

# get the first track
track = gpx_data.tracks[0]

# get the first segment
segment = track.segments[0]

rx_location = gpxpy.gpx.GPXWaypoint(44.330008, 14.697141)

distances = []
for point in segment.points:
    distance = rx_location.distance_2d(point)
    distances.append(distance)

distances = np.array(distances)
t_distances = np.linspace(0, len(data)/fs, len(distances))

data final shape: (51597000,) of duration: 2340.0 seconds


In [51]:
nperseg = 1000
overlap = 0.5
window = 'hanning'
dc = 100
crop_freq = 8000
norm_size = 5
s2g_quantization_levels = 3
s2g_mode = 'wasserstein'
welch_threshold = 0.034
s2g_threshold = 0.3
reference_model_threshold = 0.5

welch_detector = ut.WelchDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)
talmon_detector = ut.TalmonDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)
s2g_detector = ut.S2GDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, mode=s2g_mode, quantization_levels=s2g_quantization_levels)

In [52]:
# _data = data[700*fs:760*fs]
# _F, _T, _Sxx, _phasogram = ut.calc_spectrogram(_data, fs=fs, nperseg=nperseg, percent_overlap=overlap, window=window, remove_dc=dc, crop_freq=crop_freq)

# _pxx, _F_pxx = welch_detector.get_feature_vector(_data)
# _K, _F_K = s2g_detector.get_feature_vector(_data)
# _td_scores, _td_F = talmon_detector.get_feature_vector(_data)

# fig = make_subplots(rows=1, cols=4, shared_yaxes=True, column_widths=[0.7, 0.1, 0.1, 0.1])
# fig.add_trace(go.Heatmap(z=_Sxx, x=_T, y=_F, colorscale='viridis', showlegend=False, showscale=False), row=1, col=1)
# fig.add_trace(go.Scatter(x=_pxx, y=_F_pxx, name='Welch Detector', marker_color='blue'), row=1, col=2)
# fig.add_trace(go.Scatter(x=_td_scores, y=_td_F, name='Talmon Detector', marker_color='green'), row=1, col=3)
# fig.add_trace(go.Scatter(x=_K, y=_F_K, name='S2G Detector', marker_color='purple'), row=1, col=4)
# fig.update_xaxes(title_text="Time (s)", row=1, col=1)
# fig.update_yaxes(title_text="Frequency (Hz)", row=1, col=1)
# fig.show(renderer="browser")


In [53]:
# sanity - check spectral power + K only for f0 = 631, 859
start_time = time.time()

F, T, Sxx, phasogram = ut.calc_spectrogram(data, fs=fs, nperseg=nperseg, percent_overlap=overlap, window=window, remove_dc=dc, crop_freq=crop_freq)

welch_time = []
f0_1_welch_scores = []
f0_2_welch_scores = []

td_time = []
f0_1_td_scores = []
f0_2_td_scores = []

s2g_time = []
f0_1_s2g_scores = []
f0_2_s2g_scores = []

f0_1 = 631  # Hz
f0_2 = 859  # Hz
step_size = 10 * fs

for si in np.arange(start=0, stop=(len(data)-60*fs), step=step_size):
    _data = data[si:si+60*fs]
    _time = (si + len(_data) / 2) // fs  # time corresponding to the center of the window

    t = time.time()
    P = welch_detector.get_score_1fb(_data, f0_1)
    welch_time.append(time.time() - t)
    f0_1_welch_scores.append(P)
    P = welch_detector.get_score_1fb(_data, f0_2)
    welch_time.append(time.time() - t)
    f0_2_welch_scores.append(P)

    t = time.time()
    TD = talmon_detector.get_score_1fb(_data, f0_1)
    td_time.append(time.time() - t)
    f0_1_td_scores.append(TD)
    TD = talmon_detector.get_score_1fb(_data, f0_2)
    td_time.append(time.time() - t)
    f0_2_td_scores.append(TD)

    t = time.time()
    K = s2g_detector.get_score_1fb(_data, f0_1)
    s2g_time.append(time.time() - t)
    f0_1_s2g_scores.append(K)
    K = s2g_detector.get_score_1fb(_data, f0_2)
    s2g_time.append(time.time() - t)
    f0_2_s2g_scores.append(K)

detectors_taxis = np.linspace(T[0], T[-1], len(f0_1_s2g_scores))
print("completed all detectors")
print(f"total time: {time.time() - start_time}")
print(f"welch time: {np.mean(welch_time)}")
print(f"td time: {np.mean(td_time)}")
print(f"s2g time: {np.mean(s2g_time)}")

completed all detectors
total time: 88.9989230632782
welch time: 0.09385935151786134
td time: 0.0935902804659124
s2g time: 0.09667753127583287


In [54]:
# fig = make_subplots(rows=5, 
#                     cols=1, 
#                     row_heights=[0.6, 0.1, 0.1, 0.1, 0.1], 
#                     vertical_spacing=0.01, 
#                     shared_xaxes=True,
#                     row_titles=["Spectrogram", "distance","Welch", "Talmon", "S2G"])

# fig.add_trace(go.Heatmap(z=Sxx, x=T, y=F, colorscale='viridis', showlegend=False, showscale=False), row=1, col=1)
# fig.add_trace(go.Scatter(x=t_distances, y=distances, name='Distance', marker_color='black', showlegend=False), row=2, col=1)
# fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_welch_scores, name=f'Welch f={f0_1} Hz', marker_color='blue'), row=3, col=1)
# fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_welch_scores, name=f'Welch f={f0_2} Hz', marker_color='green'), row=3, col=1)
# fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_td_scores, name=f'Talmon f={f0_1} Hz', marker_color='blue', showlegend=False), row=4, col=1)
# fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_td_scores, name=f'Talmon f={f0_2} Hz', marker_color='green', showlegend=False), row=4, col=1)
# fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_s2g_scores, name=f'Score f={f0_1} Hz', marker_color='blue'), row=5, col=1)
# fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_s2g_scores, name=f'Score f={f0_2} Hz', marker_color='green'), row=5, col=1)


# fig.update_yaxes(title_text="Frequency (Hz)", row=1, col=1)
# fig.update_yaxes(title_text="Dist. (m)", row=2, col=1)
# fig.update_yaxes(title_text="Score", row=3, col=1)
# fig.update_yaxes(title_text="Score", row=4, col=1)
# fig.update_yaxes(title_text="Score", row=5, col=1)
# fig.update_xaxes(title_text="Time (s)", row=5, col=1)

# fig.update_layout(height=800, width=1200, title_text=f"Scooter experiment 600m, scores for f0 = {f0_1}, {f0_2} Hz")
# fig.show(renderer="browser")

---
## Add Noise

In [55]:
# BG Noise WAVs
folder_path = '../data/croatia/2507_1_1000m'
file_list = os.listdir(folder_path)
file_list = [f for f in file_list if f.endswith('.wav')]
file_list.sort()

bg_data = []
for file in file_list[:5]:
    _data, fs = librosa.load(os.path.join(folder_path, file))
    bg_data.append(_data)
bg_data = np.concatenate(bg_data)
print(f"bg_data final shape: {bg_data.shape} of duration: {len(bg_data)/fs} seconds")

# Add BG Noise to data
noise_factor = 10
bg_in_shape = np.concatenate([bg_data, bg_data, bg_data, bg_data, bg_data, bg_data, bg_data, bg_data])
data = data + noise_factor * bg_in_shape[:len(data)]
print(f"data final shape: {data.shape} of duration: {len(data)/fs} seconds")

bg_data final shape: (6615000,) of duration: 300.0 seconds
data final shape: (51597000,) of duration: 2340.0 seconds


In [56]:
F, T, Sxx, phasogram = ut.calc_spectrogram(data, fs=fs, nperseg=nperseg, percent_overlap=overlap, window=window, remove_dc=dc, crop_freq=crop_freq)

f0_1_welch_scores_wn = []
f0_2_welch_scores_wn = []

f0_1_td_scores_wn = []
f0_2_td_scores_wn = []

f0_1_s2g_scores_wn = []
f0_2_s2g_scores_wn = []

f0_1 = 631  # Hz
f0_2 = 859  # Hz
step_size = 10 * fs

for si in np.arange(start=0, stop=(len(data)-60*fs), step=step_size):
    _data = data[si:si+60*fs]
    _time = (si + len(_data) / 2) // fs  # time corresponding to the center of the window

    P = welch_detector.get_score_1fb(_data, f0_1)
    f0_1_welch_scores_wn.append(P)
    P = welch_detector.get_score_1fb(_data, f0_2)
    f0_2_welch_scores_wn.append(P)

    TD = talmon_detector.get_score_1fb(_data, f0_1)
    f0_1_td_scores_wn.append(TD)
    TD = talmon_detector.get_score_1fb(_data, f0_2)
    f0_2_td_scores_wn.append(TD)

    K = s2g_detector.get_score_1fb(_data, f0_1)
    f0_1_s2g_scores_wn.append(K)
    K = s2g_detector.get_score_1fb(_data, f0_2)
    f0_2_s2g_scores_wn.append(K)

detectors_taxis = np.linspace(T[0], T[-1], len(f0_1_s2g_scores))
print("completed all detectors")

completed all detectors


In [ ]:
fig = make_subplots(rows=4, 
                    cols=2, 
                    row_heights=[0.7, 0.1, 0.1, 0.1], 
                    vertical_spacing=0.01, 
                    shared_xaxes=True,
                    column_titles=["No Noise", "SNR 1:10"],
                    row_titles=["Spectrogram", "distance","Welch", "Talmon", "S2G"])

_ds = max(1, Sxx.shape[1] // 1000)
fig.add_trace(go.Heatmap(z=Sxx[:, ::_ds], x=T[::_ds], y=F, colorscale='viridis', showlegend=False, showscale=False), row=1, col=1)
fig.add_trace(go.Scatter(x=t_distances, y=distances, name='Distance', marker_color='black', showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_td_scores, name=f'Score f={f0_1} Hz', marker_color='blue'), row=3, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_td_scores, name=f'Score f={f0_2} Hz', marker_color='green'), row=3, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_s2g_scores, name=f'Score f={f0_1} Hz', marker_color='blue', showlegend=False), row=4, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_s2g_scores, name=f'Score f={f0_2} Hz', marker_color='green', showlegend=False), row=4, col=1)

dix_nn_1 = np.where(detectors_taxis >= 154)[0][0]
dix_nn_2 = np.where(detectors_taxis >= 154)[0][0]
fig.add_vline(x=detectors_taxis[dix_nn_1] , line_dash="dash", line_color="red", line_width=2, name="first detection", showlegend=True, row=3, col=1)
fig.add_vline(x=detectors_taxis[dix_nn_2], line_dash="dash", line_color="red", line_width=2, row=4, col=1)

fig.add_trace(go.Heatmap(z=Sxx[:, ::_ds], x=T[::_ds], y=F, colorscale='viridis', showlegend=False, showscale=False), row=1, col=2)
fig.add_trace(go.Scatter(x=t_distances, y=distances, name='Distance', marker_color='black', showlegend=False), row=2, col=2)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_td_scores_wn, name=f'Score f={f0_1} Hz', marker_color='blue', showlegend=False), row=3, col=2)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_td_scores_wn, name=f'Score f={f0_2} Hz', marker_color='green', showlegend=False), row=3, col=2)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_s2g_scores_wn, name=f'Score f={f0_1} Hz', marker_color='blue', showlegend=False), row=4, col=2)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_s2g_scores_wn, name=f'Score f={f0_2} Hz', marker_color='green', showlegend=False), row=4, col=2)

dix_nn_1 = np.where(detectors_taxis >= 814)[0][0]
dix_nn_2 = np.where(detectors_taxis >= 505)[0][0]
fig.add_vline(x=detectors_taxis[dix_nn_1] , line_dash="dash", line_color="red", line_width=2, row=3, col=2)
fig.add_vline(x=detectors_taxis[dix_nn_2], line_dash="dash", line_color="red", line_width=2, row=4, col=2)


fig.update_yaxes(title_text="Frequency (Hz)", row=1, col=1)
fig.update_yaxes(title_text="Dist. (m)", row=2, col=1)
fig.update_yaxes(title_text="Score", row=3, col=1)
fig.update_yaxes(title_text="Score", row=4, col=1)
fig.update_xaxes(title_text="Time (s)", row=4, col=1)
fig.update_yaxes(title_text="Frequency (Hz)", row=1, col=2)
fig.update_yaxes(title_text="Dist. (m)", row=2, col=2)
fig.update_yaxes(title_text="Score", row=3, col=2)
fig.update_yaxes(title_text="Score", row=4, col=2)
fig.update_xaxes(title_text="Time (s)", row=4, col=2)

fig.update_layout(height=800, width=1600, title_text=f"Scooter experiment 600m, scores for f0 = {f0_1}, {f0_2} Hz")
# fig.show(renderer="browser")
fig.write_image("../results/plots/nb88_600m_w_noise.pdf")

---

## Test Q levels

In [58]:
# WAV
folder_path = '../data/croatia/2407_1_600m'
file_list = os.listdir(folder_path)
file_list = [f for f in file_list if f.endswith('.wav')]
file_list.sort()

data = []
for file in file_list[6:-15]:
    # print(f"Loading file: {file}")
    _data, fs = librosa.load(os.path.join(folder_path, file))
    data.append(_data)
data = np.concatenate(data)
# data = data[80*fs:]
print(f"data final shape: {data.shape} of duration: {len(data)/fs} seconds")


# GPX
gpx_file = os.path.join(folder_path, '600m_route.gpx')
gpx_data = gpxpy.parse(open(gpx_file))

# get the first track
track = gpx_data.tracks[0]

# get the first segment
segment = track.segments[0]

rx_location = gpxpy.gpx.GPXWaypoint(44.330008, 14.697141)

distances = []
for point in segment.points:
    distance = rx_location.distance_2d(point)
    distances.append(distance)

distances = np.array(distances)
t_distances = np.linspace(0, len(data)/fs, len(distances))

data final shape: (51597000,) of duration: 2340.0 seconds


In [59]:
# BG Noise WAVs
folder_path = '../data/croatia/2507_1_1000m'
file_list = os.listdir(folder_path)
file_list = [f for f in file_list if f.endswith('.wav')]
file_list.sort()

bg_data = []
for file in file_list[:5]:
    _data, fs = librosa.load(os.path.join(folder_path, file))
    bg_data.append(_data)
bg_data = np.concatenate(bg_data)
print(f"bg_data final shape: {bg_data.shape} of duration: {len(bg_data)/fs} seconds")

# Add BG Noise to data
noise_factor = 10
bg_in_shape = np.concatenate([bg_data, bg_data, bg_data, bg_data, bg_data, bg_data, bg_data, bg_data])
data = data + noise_factor * bg_in_shape[:len(data)]
print(f"data final shape: {data.shape} of duration: {len(data)/fs} seconds")

bg_data final shape: (6615000,) of duration: 300.0 seconds
data final shape: (51597000,) of duration: 2340.0 seconds


In [60]:
nperseg = 1000
overlap = 0.5
window = 'hanning'
dc = 100
crop_freq = 8000
norm_size = 5
s2g_mode = 'wasserstein'
welch_threshold = 0.034
s2g_threshold = 0.3
reference_model_threshold = 0.5

s2g2 = ut.S2GDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, mode=s2g_mode, quantization_levels=2)
s2g3 = ut.S2GDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, mode=s2g_mode, quantization_levels=3)
s2g5 = ut.S2GDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, mode=s2g_mode, quantization_levels=5)
s2g10 = ut.S2GDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, mode=s2g_mode, quantization_levels=10)
s2g20 = ut.S2GDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, mode=s2g_mode, quantization_levels=20)

In [61]:
F, T, Sxx, phasogram = ut.calc_spectrogram(data, fs=fs, nperseg=nperseg, percent_overlap=overlap, window=window, remove_dc=dc, crop_freq=crop_freq)

f0_1_s2g_2 = []
f0_2_s2g_2 = []

f0_1_s2g_3 = []
f0_2_s2g_3 = []

f0_1_s2g_5 = []
f0_2_s2g_5 = []

f0_1_s2g_10 = []
f0_2_s2g_10 = []

f0_1_s2g_20 = []
f0_2_s2g_20 = []

f0_1 = 631  # Hz
f0_2 = 859  # Hz
step_size = 10 * fs

for si in np.arange(start=0, stop=(len(data)-60*fs), step=step_size):
    _data = data[si:si+60*fs]
    _time = (si + len(_data) / 2) // fs  # time corresponding to the center of the window

    K = s2g2.get_score_1fb(_data, f0_1)
    f0_1_s2g_2.append(K)
    K = s2g2.get_score_1fb(_data, f0_2)
    f0_2_s2g_2.append(K)

    K = s2g3.get_score_1fb(_data, f0_1)
    f0_1_s2g_3.append(K)
    K = s2g3.get_score_1fb(_data, f0_2)
    f0_2_s2g_3.append(K)

    K = s2g5.get_score_1fb(_data, f0_1)
    f0_1_s2g_5.append(K)
    K = s2g5.get_score_1fb(_data, f0_2)
    f0_2_s2g_5.append(K)

    K = s2g10.get_score_1fb(_data, f0_1)
    f0_1_s2g_10.append(K)
    K = s2g10.get_score_1fb(_data, f0_2)
    f0_2_s2g_10.append(K)

    K = s2g20.get_score_1fb(_data, f0_1)
    f0_1_s2g_20.append(K)
    K = s2g20.get_score_1fb(_data, f0_2)
    f0_2_s2g_20.append(K)

detectors_taxis = np.linspace(T[0], T[-1], len(f0_1_s2g_scores))
print("completed all detectors")

completed all detectors


In [62]:
def _parse_plotly_color(color_str):
    color_str = color_str.strip()
    if color_str.startswith('rgb'):
        return [int(c) for c in color_str[4:-1].split(',')]
    elif color_str.startswith('#'):
        h = color_str.lstrip('#')
        return [int(h[i:i+2], 16) for i in (0, 2, 4)]
    raise ValueError(f"Unknown color format: {color_str}")

def fast_to_rgb(data, colorscale_name='Viridis'):
    # Normalize data to 0-1
    d_min, d_max = data.min(), data.max()
    norm = (data - d_min) / (d_max - d_min)
    
    # Get the colorscale as a list of RGB strings/tuples
    # We use px.colors.get_colorscale to get the raw list
    scale = px.colors.get_colorscale(colorscale_name)
    
    # Extract just the RGB values into a (N, 3) array
    # Handles both 'rgb(r,g,b)' and '#rrggbb' color formats
    colors = np.array([_parse_plotly_color(s[1]) for s in scale])
    
    # Create the interpolation points (0.0 to 1.0)
    x_points = np.linspace(0, 1, len(colors))
    
    # Vectorized interpolation for R, G, and B channels
    r = np.interp(norm, x_points, colors[:, 0])
    g = np.interp(norm, x_points, colors[:, 1])
    b = np.interp(norm, x_points, colors[:, 2])
    
    # Stack into a (Rows, Cols, 3) uint8 array
    return np.dstack([r, g, b]).astype(np.uint8)

Sxx_as_image = fast_to_rgb(Sxx)

In [63]:
fig = make_subplots(rows=7, 
                    cols=1, 
                    row_heights=[0.6, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1], 
                    vertical_spacing=0.01, 
                    shared_xaxes=True,
                    row_titles=["Spectrogram", "Distance", "Q=2","Q=3","Q=5","Q=10","Q=20"])

_ds = max(1, Sxx.shape[1] // 1000)
fig.add_trace(go.Heatmap(z=Sxx[:, ::_ds], x=T[::_ds], y=F, colorscale='viridis', showlegend=False, showscale=False), row=1, col=1)
fig.add_trace(go.Scatter(x=t_distances, y=distances, name='Distance', marker_color='black', showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_s2g_2, name=f'Score f={f0_1} Hz', marker_color='blue'), row=3, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_s2g_2, name=f'Score f={f0_2} Hz', marker_color='green'), row=3, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_s2g_3, name=f'Score f={f0_1} Hz', marker_color='blue', showlegend=False), row=4, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_s2g_3, name=f'Score f={f0_2} Hz', marker_color='green', showlegend=False), row=4, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_s2g_5, name=f'Score f={f0_1} Hz', marker_color='blue', showlegend=False), row=5, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_s2g_5, name=f'Score f={f0_2} Hz', marker_color='green', showlegend=False), row=5, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_s2g_10, name=f'Score f={f0_1} Hz', marker_color='blue', showlegend=False), row=6, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_s2g_10, name=f'Score f={f0_2} Hz', marker_color='green', showlegend=False), row=6, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_s2g_20, name=f'Score f={f0_1} Hz', marker_color='blue', showlegend=False), row=7, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_s2g_20, name=f'Score f={f0_2} Hz', marker_color='green', showlegend=False), row=7, col=1)

fig.update_yaxes(title_text="Frequency (Hz)", row=1, col=1)
fig.update_yaxes(title_text="Dist. (m)", row=2, col=1)
fig.update_yaxes(title_text="Score", row=3, col=1)
fig.update_yaxes(title_text="Score", row=4, col=1)
fig.update_yaxes(title_text="Score", row=5, col=1)
fig.update_yaxes(title_text="Score", row=6, col=1)
fig.update_yaxes(title_text="Score", row=7, col=1)
fig.update_xaxes(title_text="Time (s)", row=7, col=1)


fig.update_layout(height=800, width=1200, title_text=f"Scooter experiment 600m, scores for f0 = {f0_1}, {f0_2} Hz")
fig.show(renderer="browser")
fig.write_image("../results/plots/nb88_600m_Qanalysis.pdf")